# Matched framework comparison

This notebook loads the small committed comparison snapshot. Re-run it with the documented comparison command when framework dependencies are installed; loading the snapshot keeps classroom execution fast and offline.

In [ ]:
import json
from pathlib import Path

result = json.loads(Path("evaluation/comparison/results/result.json").read_text())
assert result["comparison_schema_version"] == "1"
[profile["implementation"] for profile in result["profiles"]]

## Canonical outcomes and calls

Task completion, schema validity, evidence metrics, call counts and trajectory validity are deterministic under the matched mock fixture. They are comparison criteria; prose wording is not.

In [ ]:
first_run = {}
for run in result["runs"]:
    first_run.setdefault(run["implementation"], run)
canonical_summary = {
    name: {
        "task_completed": run["metrics"]["task_completed"],
        "final_answer_valid": run["metrics"]["final_answer_schema_valid"],
        "model_calls": run["metrics"]["model_calls"],
        "tool_calls": run["metrics"]["tool_calls"],
        "steps": run["total_steps"],
        "framework_events": run["framework_specific_trace_events"],
    }
    for name, run in first_run.items()
}
canonical_summary

Framework-specific trace events expose orchestration semantics: plain Python adds none, while graph nodes, CrewAI stages and SDK handoffs are retained as canonical event payloads. No framework-native object enters the final state or output.

In [ ]:
event_kinds = {name: run["framework_event_kinds"] for name, run in first_run.items()}
assert all(summary["model_calls"] == 4 for summary in canonical_summary.values())
assert all(summary["tool_calls"] == 3 for summary in canonical_summary.values())
event_kinds

## Caveats

Latency, peak memory, dependency footprint, startup overhead and source size are machine-dependent observations—not deterministic equality fields and not an overall ranking. The comparison disables autonomous CrewAI delegation and memory and excludes the OpenAI Agents SDK autonomous Runner so that hidden calls do not invalidate the matched design. LangGraph uses native in-memory checkpoints inside a run and canonical JSON checkpoints for durable resumption.

In [ ]:
observations = {
    profile["implementation"]: {
        "dependency_count": profile["dependency_footprint"]["transitive_distribution_count"],
        "orchestration_lines": profile["orchestration_nonblank_noncomment_lines"],
        "limitation": profile["limitation"],
    }
    for profile in result["profiles"]
}
observations